In [2]:
import numpy as np
import rasterio as rio
from rasterio.plot import show
import pandas as pd
from pathlib import Path
from scipy.stats import linregress
import matplotlib.pyplot as plt
import seaborn as sns
import cmocean

In [3]:
landmask_file = rio.open("C:\\Users\\simon\\OneDrive - University of Copenhagen\\Documents\\Arbeitsmappe\\GEMLST\\GEMLST\\GEMLST_MODIS\\Data\\Test_Gapfilling\\landmask.tif")
landmask = landmask_file.read()
lst_example = rio.open("C:\\Users\\simon\\OneDrive - University of Copenhagen\\Documents\\Arbeitsmappe\\GEMLST\\GEMLST\\GEMLST_MODIS\\Data\\Test_Gapfilling\\GEMLST_MODIS_20000315.tif")

In [9]:
# SATLST Data: 
# satfolder = "/home/jovyan/work/AVOCA/GEM/Development/Data/BatchExport_LST/YYYY/GEMLST_MODIS_yyyymmdd.tif"
satfolder = "./Data/Test_Gapfilling/YYYY/GEMLST_MODIS_yyyymmdd.tif"
era5folder = "./Data/Test_Gapfilling/GL1000m_reproj/"

# List all files in the folder and sort them
imfiles = sorted(Path(era5folder).glob("*.tif"))

# slice inclusive range
# imfiles = imfiles[0:1000]

# result paths
results_total = "./ERA5_SATLST_total.h5"
results_ice = "./ERA5_SATLST_ice.h5"
results_land = "./ERA5_SATLST_land.h5"

In [ ]:
for imfile in imfiles:

### FILE IMPORT AND MATCHING ###
    # Find the corresponding SATLST image
    yearstring = imfile.stem.split("_")[2]
    year = pd.to_datetime(yearstring, format="%Y")
    doy = imfile.stem.split("_")[3][1:]
    date = year + pd.to_timedelta(int(doy) - 1, unit="D")
    date = date.strftime("%Y-%m-%d")
    datestring = date.replace("-", "")
    imagepath = satfolder.replace('YYYY', yearstring).replace('yyyymmdd', datestring)


### CHANGE TO WITH X OPEN XXX ###
    sat_file = rio.open(imagepath)
    sat_data = sat_file.read(1)  
    # Set values to NaN where qa band is 0 (no sat obs)
    sat_na = np.where(sat_file.read(2) == 0, np.nan, sat_data)
    # also here with x open  ... 
    era5_file = rio.open(imfile)
    era5_data = era5_file.read(1)


### ARRANGE ARRAYS ####
    # Transform images into a 1D array (numpy ravel)
    era5_1d = era5_data.ravel()
    satlst_1d = sat_na.ravel()
    landmask_1d = landmask.ravel()
    # print(f'ERA5 1D shape: {era5_1d.shape}, SATLST 1D shape: {satlst_1d.shape}, Landmask 1D shape: {landmask_1d.shape}')


### MASKING AND NA HANDLING ###
    na_mask = ~np.isnan(era5_1d) & ~np.isnan(satlst_1d)
    na_mask_ice = ~np.isnan(era5_1d) & ~np.isnan(satlst_1d) & (landmask_1d == 0)
    na_mask_land = ~np.isnan(era5_1d) & ~np.isnan(satlst_1d) & (landmask_1d == 1)

    era5_1d_nona = era5_1d[na_mask][:10]
    satlst_1d_nona = satlst_1d[na_mask][:10]
    era5_1d_nona_ice = era5_1d[na_mask_ice][:10]
    satlst_1d_nona_ice = satlst_1d[na_mask_ice][:10]
    era5_1d_nona_land = era5_1d[na_mask_land][:10]
    satlst_1d_nona_land = satlst_1d[na_mask_land][:10]
    
### SAVE FILE ### 

    # # Append to HDF5 files
    # df_total = pd.DataFrame({'date': [date] * len(era5_1d_nona), 'era5': era5_1d_nona, 'satlst': satlst_1d_nona})
    # df_total.to_hdf(results_total, key="data", mode="a", append=True, index=False)

    # df_ice = pd.DataFrame({'date': [date] * len(era5_1d_nona_ice), 'era5': era5_1d_nona_ice, 'satlst': satlst_1d_nona_ice})
    # df_ice.to_hdf(results_ice, key="data", mode="a", append=True, index=False)

    # df_land = pd.DataFrame({'date': [date] * len(era5_1d_nona_land), 'era5': era5_1d_nona_land, 'satlst': satlst_1d_nona_land})
    # df_land.to_hdf(results_land, key="data", mode="a", append=True, index=False)

### PRINTS###
    # Every round
    print(f'File {imfile} processed, corresponding to date {date}. Total valid points: {len(era5_1d_nona)}, Ice valid points: {len(era5_1d_nona_ice)}, Land valid points: {len(era5_1d_nona_land)}')


# Confirm loop completion
print('ALL FILES DONE') 


[] 
 [] 
 []
File Data\Test_Gapfilling\GL1000m_reproj\t2m_1000m_2000_d001.tif processed, corresponding to date 2000-01-01. Total valid points: 0, Ice valid points: 0, Land valid points: 0
[] 
 [] 
 []
File Data\Test_Gapfilling\GL1000m_reproj\t2m_1000m_2000_d002.tif processed, corresponding to date 2000-01-02. Total valid points: 0, Ice valid points: 0, Land valid points: 0
[] 
 [] 
 []
File Data\Test_Gapfilling\GL1000m_reproj\t2m_1000m_2000_d003.tif processed, corresponding to date 2000-01-03. Total valid points: 0, Ice valid points: 0, Land valid points: 0
ALL FILES DONE


In [ ]:
### LINEAR REGRESSION ### 
    # print(f'Image:{date}')

    slope, intercept, r_value, p_value, std_err = linregress(era5_1d_nona, satlst_1d_nona)
    # print(f'Total\tslope: {round(slope, 3)}, intercept: {round(intercept, 3)}, r_value: {round(r_value, 2)}, p_value: {round(p_value,3)}, std_err: {round(std_err, 3)}')

    slope_i, intercept_i, r_value_i, p_value_i, std_err_i = linregress(era5_1d_nona_ice, satlst_1d_nona_ice)
    # print(f'Ice\tslope: {round(slope_i, 3)}, intercept: {round(intercept_i, 3)}, r_value: {round(r_value_i, 2)}, p_value: {round(p_value_i,3)}, std_err: {round(std_err_i, 3)}')

    slope_l, intercept_l, r_value_l, p_value_l, std_err_l = linregress(era5_1d_nona_land, satlst_1d_nona_land)
    # print(f'Land\tslope: {round(slope_l, 3)}, intercept: {round(intercept_l, 3)}, r_value: {round(r_value_l, 2)}, p_value: {round(p_value_l,3)}, std_err: {round(std_err_l, 3)}\n')


### PLOT (for test purposes) 

    fig, axes = plt.subplots(1, 3, figsize=(12, 6))
    xlims = (era5_1d_nona.min(), era5_1d_nona.max())
    ylims = (satlst_1d_nona.min(), satlst_1d_nona.max())
    
    # Plot 1 total
    ax = axes[0]
    hexbin = ax.hexbin(era5_1d_nona, satlst_1d_nona,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona, y=satlst_1d_nona, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], 'k--', alpha=0.5, label='1:1 line')
    ax.set_title('Total')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona.shape[0]}\n'
    stats_text += f'R² = {r_value**2:.3f}\n'
    stats_text += f'Slope = {slope:.3f}\n'
    stats_text += f'Intercept = {intercept:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))


    # Plot 2 Ice
    ax = axes[1]
    hexbin = ax.hexbin(era5_1d_nona_ice, satlst_1d_nona_ice,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona_ice, y=satlst_1d_nona_ice, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], 'k--', alpha=0.5, label='1:1 line')
    ax.set_title('Ice')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona_ice.shape[0]}\n'
    stats_text += f'R² = {r_value_i**2:.3f}\n'
    stats_text += f'Slope = {slope_i:.3f}\n'
    stats_text += f'Intercept = {intercept_i:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Plot 3 Land
    ax = axes[2]
    hexbin = ax.hexbin(era5_1d_nona_land, satlst_1d_nona_land,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona_land, y=satlst_1d_nona_land, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], alpha=0.5, label='1:1 line')
    ax.set_title('Land')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona_land.shape[0]}\n'
    stats_text += f'R² = {r_value_l**2:.3f}\n'
    stats_text += f'Slope = {slope_l:.3f}\n'
    stats_text += f'Intercept = {intercept_l:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    fig.suptitle(f'ERA5 vs SATLST on {date}', fontsize=16)
    plt.tight_layout()
    plt.show()